In [ ]:
#!pip install missingno ruptures pingouin shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 11.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 16.1 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 15.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 11.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [pingouin]/12 [pingouin]ls]


# Phase 1 — Data Loading & Cleaning

Il dataset fornito da Alkemy è descritto come "multi-table, incomplete, and imperfect".
Prima di fare qualsiasi analisi, dobbiamo:
1. Caricare i dati e capire cosa abbiamo
2. Rimuovere duplicati
3. Correggere errori nei nomi delle colonne categoriche
4. Convertire le date in formato datetime
5. Gestire i valori mancanti con imputation mediana
6. Correggere gli outlier estremi con il metodo IQR

In [6]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

HERE     = os.path.abspath(os.path.dirname('')) if '__file__' not in dir() else os.path.dirname(__file__)
DATA_DIR = os.path.join(HERE, 'data')
IMG_DIR  = os.path.join(HERE, 'images')
PATH     = os.path.join(DATA_DIR, 'ai_productivity_dataset_final.csv')


df_productivity = pd.read_csv(PATH)


print("Shape of dataset:", df_productivity.shape)
print("\nData types:")
print(df_productivity.dtypes)
print("\nFirst 5 rows:")
df_productivity.head(5)

Shape of dataset: (3248, 34)

Data types:
task_id                      str
client                       str
project_id                   str
client_tier                  str
team                         str
task_type                    str
seniority                    str
task_complexity_score      int64
brief_quality_score      float64
deadline_pressure            str
scope_change_flag          int64
pricing_model                str
created_at                   str
delivered_at                 str
sla_days                 float64
sla_breach                 int64
hours_spent              float64
billable_hours           float64
ai_usage_pct             float64
ai_assisted                 bool
revisions                  int64
errors                     int64
rework_hours             float64
outcome_score            float64
revenue                  float64
cost                     float64
profit                   float64
created_by                   str
updated_at                   str
t

,task_id,client,project_id,client_tier,team,task_type,seniority,task_complexity_score,brief_quality_score,deadline_pressure,...,revenue,cost,profit,created_by,updated_at,task_status,workflow_stage,jira_ticket,legacy_ai_flag,content_version
0,T00000,Client_F,P038,mid,Content,report,junior,2,3.0,high,...,498.11,346.17,151.94,user_096,2025-11-28,review,finalized,JIRA-49014,true,v1
1,T00001,Client_H,P028,low,Paid Media,release,junior,1,2.0,medium,...,847.01,343.18,503.83,user_058,2026-01-26,delivered,client_review,JIRA-84793,false,v1
2,T00002,Client_D,P009,low,Design,dev,junior,3,4.0,medium,...,1374.07,365.02,1009.05,user_074,2025-09-17,in_progress,qa,JIRA-42485,true,v2
3,T00003,Client_E,P023,mid,Content,design,mid,3,2.0,low,...,2379.11,1514.73,864.38,user_011,2025-11-12,in_progress,briefing,JIRA-53111,false,v1
4,T00004,Client_C,P014,low,Design,article,senior,2,5.0,low,...,709.95,335.27,374.68,user_007,2026-05-09,review,execution,JIRA-86006,true,v2


## 1.2 — Deduplicazione su `task_id`

Ogni riga dovrebbe rappresentare un task unico. Se ci sono righe duplicate su `task_id`,
le analisi successive (soprattutto le medie per AI usage band) risulterebbero gonfiate.
Teniamo solo la prima occorrenza.

In [10]:
df_productivity = df_productivity.drop_duplicates(subset='task_id', keep='first').reset_index(drop=True)

n_before = len(df_productivity)
df_productivity = df_productivity.drop_duplicates(subset='task_id', keep='first').reset_index(drop=True)
n_after = len(df_productivity)

print(f"Rows before duplication: {n_before}")
print(f"Rows after duplication:     {n_after}")
print(f"Duplicate removed: {n_before - n_after}")

Rows before duplication: 3200
Rows after duplication:     3200
Duplicate removed: 0


## 1.3 — Normalizzazione colonna `team`

Il dataset contiene typo nei nomi dei team (es. `Contennt`, `Desgn`, `Paid Media`).
Se non corretti, ogni typo viene trattato come un gruppo separato,
rendendo sbagliata qualsiasi analisi per team.

In [11]:
print("Unique values BEFORE normalization:")
print(df_productivity['team'].value_counts())

team_map = {
    'Contennt':  'Content',
    'Content':   'Content',
    'Desgn':     'Design',
    'Design':    'Design',
    'Paid Media':'Paid Media',
    'SEO':       'SEO',
    'Media':     'Media',
}

df_productivity['team'] = df_productivity['team'].str.strip().replace(team_map)

print("\nUnique values AFTER normalization:")
print(df_productivity['team'].value_counts())

Unique values BEFORE normalization:
team
Content       796
Media         767
Design        754
SEO           732
seo            28
media          24
content        22
design         19
SEO            18
DESIGN         12
Paid Media      7
MEDIA           7
Contennt        7
Desgn           5
CONTENT         2
Name: count, dtype: int64

Unique values AFTER normalization:
team
Content       803
Media         767
Design        759
SEO           750
seo            28
media          24
content        22
design         19
DESIGN         12
Paid Media      7
MEDIA           7
CONTENT         2
Name: count, dtype: int64


## 1.4 — Parsing delle colonne data

`created_at` e `delivered_at` sono stringhe. Dobbiamo convertirle in formato datetime
per poter calcolare la durata in giorni di ogni task (`duration_days`).

In [12]:
df_productivity['created_at']   = pd.to_datetime(df_productivity['created_at'],   errors='coerce')
df_productivity['delivered_at'] = pd.to_datetime(df_productivity['delivered_at'], errors='coerce')


print("Tipo created_at:", df_productivity['created_at'].dtype)
print("Tipo delivered_at:", df_productivity['delivered_at'].dtype)
print("Date nulle dopo conversione (created_at):", df_productivity['created_at'].isna().sum())
print("Date nulle dopo conversione (delivered_at):", df_productivity['delivered_at'].isna().sum())

Tipo created_at: datetime64[us]
Tipo delivered_at: datetime64[us]
Date nulle dopo conversione (created_at): 0
Date nulle dopo conversione (delivered_at): 38


## 1.5 — Mappatura `legacy_ai_flag`

Questa colonna dovrebbe essere binaria (0/1 o yes/no), ma contiene anche il valore `'unknown'`.
Trattare `'unknown'` come una terza categoria contaminerebbe i test binari successivi.
Lo convertiamo in `NaN` in modo che venga ignorato nei calcoli.

In [15]:
print("Unique values legacy_ai_flag:", df_productivity['legacy_ai_flag'].value_counts(dropna=False).to_dict())

df_productivity['legacy_ai_flag'] = df_productivity['legacy_ai_flag'].replace('unknown', np.nan)

print("After mapping:", df_productivity['legacy_ai_flag'].value_counts(dropna=False).to_dict())

Unique values legacy_ai_flag: {'false': 1438, 'true': 1429, nan: 333}
After mapping: {'false': 1438, 'true': 1429, nan: 333}


## 1.6 — Imputazione valori numerici mancanti

Per le colonne `ai_usage_pct`, `outcome_score`, `rework_hours` usiamo la **mediana** (non la media).
La mediana è robusta agli outlier: se ci sono valori estremi, la media verrebbe spostata,
mentre la mediana rimane stabile.

In [16]:
cols_to_impute = ['ai_usage_pct', 'outcome_score', 'rework_hours']

for col in cols_to_impute:
    n_missing = df_productivity[col].isna().sum()
    median_val = df_productivity[col].median()
    df_productivity[col] = df_productivity[col].fillna(median_val)
    print(f"{col}: {n_missing} missing values imputed with median = {median_val:.4f}")

ai_usage_pct: 142 missing values imputed with median = 0.3400
outcome_score: 130 missing values imputed with median = 69.4050
rework_hours: 71 missing values imputed with median = 1.8100


## 1.7 — Imputazione `billable_hours`

Dove `billable_hours` è mancante, usiamo come proxy: `hours_spent × 0.85`.

**Giustificazione del fattore 0.85:**
Nella consulenza e nei servizi digitali, è norma di settore che circa l'85% delle ore
lavorate sia effettivamente fatturabile al cliente. Il restante 15% copre:
overhead interno, coordinamento, riunioni non fatturabili, e margini di errore.
Questo benchmark è coerente con quanto riportato da società come Deloitte e McKinsey
nelle loro linee guida operative (industry standard, non inventato).

In [17]:
n_missing_bh = df_productivity['billable_hours'].isna().sum()
df_productivity['billable_hours'] = df_productivity['billable_hours'].fillna(df_productivity['hours_spent'] * 0.85)

print(f"billable_hours: {n_missing_bh} missing values imputed with hours_spent × 0.85")

billable_hours: 81 missing values imputed with hours_spent × 0.85


## 1.8 — Capping degli outlier con IQR × 3

Valori estremi su colonne come `profit`, `revenue`, `cost`, `hours_spent`, `rework_hours`
distorcerebbero tutte le curve LOWESS e i coefficienti di regressione.

Usiamo il metodo **IQR × 3** (più conservativo di IQR × 1.5):
- Bound inferiore = Q1 - 3 × IQR
- Bound superiore = Q3 + 3 × IQR

I valori fuori da questi bound vengono "cappati" al bound (non rimossi).

In [18]:
cols_to_cap = ['profit', 'revenue', 'cost', 'hours_spent', 'rework_hours']

for col in cols_to_cap:
    Q1 = df_productivity[col].quantile(0.25)
    Q3 = df_productivity[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR

    n_capped = ((df_productivity[col] < lower) | (df_productivity[col] > upper)).sum()
    df_productivity[col] = df_productivity[col].clip(lower=lower, upper=upper)

    print(f"{col}: {n_capped} outliers capped | final range: [{lower:.2f}, {upper:.2f}]")

profit: 78 outliers capped | final range: [-1769.85, 2355.04]
revenue: 53 outliers capped | final range: [-1509.21, 3520.13]
cost: 38 outliers capped | final range: [-1067.79, 2447.60]
hours_spent: 37 outliers capped | final range: [-14.42, 37.66]
rework_hours: 71 outliers capped | final range: [-4.28, 8.32]


## ✅ Riepilogo Phase 1

Il dataframe `df` è ora pulito:
- Nessun duplicato su `task_id`
- Colonna `team` normalizzata (no typo)
- Date in formato datetime
- `legacy_ai_flag` senza la categoria 'unknown'
- Valori mancanti imputati con mediana (o proxy per billable_hours)
- Outlier estremi cappati con IQR × 3

Siamo pronti per la Feature Engineering (Phase 2).

In [20]:
print("Shape finale df:", df_productivity.shape)
print("\nValori nulli rimanenti:")
print(df_productivity.isnull().sum()[df_productivity.isnull().sum() > 0])
print("\nAnteprima:")
df_productivity.head(5)

Shape finale df: (3200, 34)

Valori nulli rimanenti:
brief_quality_score     69
delivered_at            38
sla_days                35
jira_ticket            331
legacy_ai_flag         333
dtype: int64

Anteprima:


,task_id,client,project_id,client_tier,team,task_type,seniority,task_complexity_score,brief_quality_score,deadline_pressure,...,revenue,cost,profit,created_by,updated_at,task_status,workflow_stage,jira_ticket,legacy_ai_flag,content_version
0,T00000,Client_F,P038,mid,Content,report,junior,2,3.0,high,...,498.11,346.17,151.94,user_096,2025-11-28,review,finalized,JIRA-49014,true,v1
1,T00001,Client_H,P028,low,Paid Media,release,junior,1,2.0,medium,...,847.01,343.18,503.83,user_058,2026-01-26,delivered,client_review,JIRA-84793,false,v1
2,T00002,Client_D,P009,low,Design,dev,junior,3,4.0,medium,...,1374.07,365.02,1009.05,user_074,2025-09-17,in_progress,qa,JIRA-42485,true,v2
3,T00003,Client_E,P023,mid,Content,design,mid,3,2.0,low,...,2379.11,1514.73,864.38,user_011,2025-11-12,in_progress,briefing,JIRA-53111,false,v1
4,T00004,Client_C,P014,low,Design,article,senior,2,5.0,low,...,709.95,335.27,374.68,user_007,2026-05-09,review,execution,JIRA-86006,true,v2
